In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

In [2]:
df_network = pd.read_excel('Network_rural.xlsx')
node_list = pd.read_excel('Nodes.xlsx')
df_demand = pd.read_excel('Demand.xlsx') #demand for 27.1.2019

In [3]:
df_demand = df_demand / 50 #turining it to pu 

In [4]:
def node_index(node):
    return(node_list[node_list['node']==node].index[0])

In [5]:
def node_name(n):
    return(node_list['node'][n])

In [6]:
def load(n):
    return(np.array(df_demand.get(node_list['node'][n], default=np.zeros(df_demand.shape[0]))))

In [7]:
node_number = len(node_list)
T = len(df_demand)
partitions = 10
v_min = 0.95
v_max = 1.05
v_nom = 1
q_factor_demand = np.tan(np.arccos(0.8))
q_factor_DG = np.tan(np.arccos(0.8))
delta_s = v_nom * df_network['IMAX'] / partitions
R = range(1,partitions+1)


In [8]:
def HC_estimator(node,max_forecast):
    #parameters
    const = []
    node_number = len(node_list)
    T = len(df_demand)
    partitions = 10
    v_min = 0.95
    v_max = 1.05
    v_nom = 1
    obj = 0
    q_factor_demand = np.tan(np.arccos(0.8))
    q_factor_DG = np.tan(np.arccos(0.8))
    delta_s = v_nom * df_network['IMAX'] / partitions
    R = range(1,partitions+1)
        # power trading variables
    power_act_poc = cp.Variable(shape = (node_number,T))                         # active power of poc
    power_react_poc = cp.Variable(shape = (node_number,T))                       # reactive power of poc
    DG_HC = cp.Variable(shape = (node_number,T), nonneg = True)                  # DG hosting capacity

    # network related parameters
    flow_act_pos = cp.Variable(shape = (len(df_network),T), nonneg = True)      # positive active power flow
    flow_act_neg = cp.Variable(shape = (len(df_network),T), nonneg = True)      # negative active power flow
    flow_react_pos = cp.Variable(shape = (len(df_network),T), nonneg = True)    # positive reactive power flow
    flow_react_neg = cp.Variable(shape = (len(df_network),T), nonneg = True)    # negative reactive power flow
    voltage_square = cp.Variable(shape = (node_number,T), nonneg = True)        # voltage square
    current_square = cp.Variable(shape = (len(df_network),T), nonneg = True)    # current square
    power_act_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        power_act_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
    power_react_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        power_react_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
    par1 = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        par1[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
    par2 = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        par2[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)

    for t in range(T):
        for n in range(node_number):
            const += [power_act_poc[n,t] # active power to the local grid
                        - (load(n)[t] )   # load of the system
                        + DG_HC[n,t]      # DG injected to the grid
                        - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
                        + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_neg)[0,t]
                        - ((np.array([df_network.fr == node_name(n)]).astype(int) 
                        * np.array(df_network.RL)) @ current_square)[0,t]
                        + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
                        - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_neg)[0,t] == 0]
            
            const += [power_react_poc[n,t] # reactive power to the local grid
                        - q_factor_demand * (load(n)[t]) # reactive power consumed from demand
                        + q_factor_DG * DG_HC[n,t] # reactive power injected from DGs
                        - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
                        + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_neg)[0,t]
                        - ((np.array([df_network.fr == node_name(n)]).astype(int) 
                        * np.array(df_network.XL)) @ current_square)[0,t]
                        + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
                        - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_neg)[0,t] == 0]
            # Voltage constraints
            const += [voltage_square[n,t] <= np.square(v_max)] #maximum voltage limit
            const += [np.square(v_min) <= voltage_square[n,t]] #minimum voltage limit
            const += [voltage_square[0,t] == np.square(v_nom)] #voltage of slack bus
            const += [DG_HC[0,t] == 0] #We don't have DG at slack bus

            # Constraints related to the poc node
            if n != 0:
                const += [power_act_poc[n,t] == 0]
                const += [power_react_poc[n,t] == 0]
                #scenarios
                const += [DG_HC[int(node),t] == float(max_forecast)/50]
            # network constraints:
        for net in range(len(df_network)):
             # voltage and power relationship:
            const += [voltage_square[node_index(df_network.loc[net,'fr']),t] -
                    voltage_square[node_index(df_network.loc[net,'to']),t] -
                    (np.square(df_network.RL[net]) + np.square(df_network.XL[net])) * current_square[net,t] -
                    2 * df_network.RL[net] * (flow_act_pos[net,t] - flow_act_neg[net,t]) -
                    2 * df_network.XL[net] * (flow_react_pos[net,t] - flow_react_neg[net,t]) == 0]

            # active and reactive power flow constraints
            const += [flow_act_pos[net,t] + flow_act_neg[net,t] <= v_nom * df_network.IMAX[net]]
            const += [flow_react_pos[net,t] + flow_react_neg[net,t] <= v_nom * df_network.IMAX[net]]

            # voltage constraint
            const += [current_square[net,t] <= np.square(df_network.IMAX[net])]
            # piece-wise linearization-related constraints
            const += [np.square(v_nom) * current_square[net,t] == 
                        sum(par1[:])[net,t] + sum(par2[:])[net,t]]

            const += [flow_act_pos[net,t] + flow_act_neg[net,t] ==
                        sum(power_act_delta[:])[net,t]]

            const += [flow_react_pos[net,t] + flow_react_neg[net,t] ==
                        sum(power_react_delta[:])[net,t]]

            for r in R:
                const += [par1[r][net,t] == (2*r-1) * delta_s[net] * power_act_delta[r][net,t]]
                const += [par2[r][net,t] == (2*r-1) * delta_s[net] * power_react_delta[r][net,t]]
                const += [power_act_delta[r][net,t] <= delta_s[net]]
                const += [power_react_delta[r][net,t] <= delta_s[net]]
        #constraints related to transformer capacity
        const += [cp.sum(power_act_poc[:,t]) <= (800/1.31) /50]
        const += [cp.sum(power_react_poc[:,t]) <= (800/1.68) /50]
        const += [cp.sum(power_act_poc[:,t]) >= -(800/1.31) /50]
        const += [cp.sum(power_react_poc[:,t]) >= -(800/1.68) /50]
        #objective function
    obj = cp.sum(cp.sum(DG_HC))
    problem = cp.Problem(cp.Maximize(obj), const)
    obj_value = problem.solve(verbose=False)
    if problem.status == 'optimal':
        DG = np.array(DG_HC.value * 50)
        HC_non_slack = DG[1:,:]
        HC_nodal = np.min(HC_non_slack, axis=1)
        HC_total = sum(HC_nodal)
        return (HC_nodal, HC_total)
    else: 
        return(np.zeros(10),0)
    
    

In [9]:
HC_before = np.array([169.54830321,  28.59989646,  20.79479701,  18.27446834,
       169.15219121,  24.31394066,  16.55142523,  13.44041565,
        11.97987025,  11.45868918])

In [10]:
#node 1
HC_1 = np.zeros(len(np.arange(int(HC_before[0]),int(HC_before[0]*240),10)))
vector_HC_1 = np.zeros((len(np.arange(HC_before[0],240,10)),10))
counter = 0
for i in np.arange(HC_before[0],240,10):
    vector_HC_1[counter,:], HC_1[counter] = HC_estimator(1,i)
    counter += 1

In [16]:
np.savetxt('HC1.csv', vector_HC_1, delimiter=',')

In [11]:
node = 2
i = 0
HC_2 = np.zeros(len(np.arange(int(HC_before[node-1]),235,10)))
vector_HC_2 = np.zeros((len(np.arange(HC_before[node-1],235,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],235,10):
    vector_HC_2[counter,:], HC_2[counter] = HC_estimator(node,i)
    counter += 1

In [17]:
np.savetxt('HC2.csv', vector_HC_2, delimiter=',')

In [12]:
node = 3
i = 0
HC_3 = np.zeros(len(np.arange(int(HC_before[node-1]),235,10)))
vector_HC_3 = np.zeros((len(np.arange(HC_before[node-1],235,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],235,10):
    vector_HC_3[counter,:], HC_3[counter] = HC_estimator(node,i)
    counter += 1

In [18]:
np.savetxt('HC3.csv', vector_HC_3, delimiter=',')

In [22]:
node = 4
i = 0
HC_4 = np.zeros(len(np.arange(int(HC_before[node-1]),235,10)))
vector_HC_4 = np.zeros((len(np.arange(HC_before[node-1],235,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],235,10):
    vector_HC_4[counter,:], HC_4[counter] = HC_estimator(node,i)
    counter += 1

In [25]:
np.savetxt('HC4.csv', vector_HC_4, delimiter=',')

In [28]:
node = 5
i = 0
HC_5 = np.zeros(len(np.arange(int(HC_before[node-1]),280,10)))
vector_HC_5 = np.zeros((len(np.arange(HC_before[node-1],280,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],280,10):
    vector_HC_5[counter,:], HC_5[counter] = HC_estimator(node,i)
    counter += 1

In [30]:
np.savetxt('HC5.csv', vector_HC_5, delimiter=',')

In [33]:
node = 6
i = 0
HC_6 = np.zeros(len(np.arange(int(HC_before[node-1]),280,10)))
vector_HC_6 = np.zeros((len(np.arange(HC_before[node-1],280,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],280,10):
    vector_HC_6[counter,:], HC_6[counter] = HC_estimator(node,i)
    counter += 1

In [35]:
np.savetxt('HC6.csv', vector_HC_6, delimiter=',')

In [36]:
node = 7
i = 0
HC_7 = np.zeros(len(np.arange(int(HC_before[node-1]),280,10)))
vector_HC_7 = np.zeros((len(np.arange(HC_before[node-1],280,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],280,10):
    vector_HC_7[counter,:], HC_7[counter] = HC_estimator(node,i)
    counter += 1

In [40]:
np.savetxt('HC7.csv', vector_HC_7, delimiter=',')

In [41]:
node = 8
i = 0
HC_8 = np.zeros(len(np.arange(int(HC_before[node-1]),280,10)))
vector_HC_8 = np.zeros((len(np.arange(HC_before[node-1],280,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],280,10):
    vector_HC_8[counter,:], HC_8[counter] = HC_estimator(node,i)
    counter += 1

In [42]:
np.savetxt('HC8.csv', vector_HC_8, delimiter=',')

In [43]:
node = 9
i = 0
HC_9 = np.zeros(len(np.arange(int(HC_before[node-1]),280,10)))
vector_HC_9 = np.zeros((len(np.arange(HC_before[node-1],280,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],280,10):
    vector_HC_9[counter,:], HC_9[counter] = HC_estimator(node,i)
    counter += 1

In [44]:
np.savetxt('HC9.csv', vector_HC_9, delimiter=',')

In [10]:
node = 10
i = 0
HC_10 = np.zeros(len(np.arange(int(HC_before[node-1]),280,10)))
vector_HC_10 = np.zeros((len(np.arange(HC_before[node-1],280,10)),10))
counter = 0
for i in np.arange(HC_before[node-1],280,10):
    vector_HC_10[counter,:], HC_10[counter] = HC_estimator(node,i)
    counter += 1

In [11]:
np.savetxt('HC10.csv', vector_HC_10, delimiter=',')